# Full-range optimized Rutherford scan

Edit only the settings in the next cell, then choose **Run All**.

This notebook is the first stage of the coupled parameter-truncation workflow. It
computes the free-space Rutherford probability that the ion recoil exceeds the
selected energy threshold and writes `rutherford_full_range_energy_scan.npz`.
The revised potential notebook loads that file and excludes points with
`P_Rutherford < 1e-6` from its heat maps.

Both notebooks scan the complete logarithmic ranges

- $10^{-27}\,\mathrm{kg} \le m_{\rm DM} \le 10^{-20}\,\mathrm{kg}$
- $10^{-3} \le \epsilon \le 10^{3}$

The speed quadrature uses the midpoint representatives of five equal-probability
Maxwell--Boltzmann bins, $q=(0.10,0.30,0.50,0.70,0.90)$.


In [ ]:
import numpy as np

# ============================================================
# USER SETTINGS FOR THE FULL-RANGE ANALYTIC RUTHERFORD SCAN
# ============================================================

SCAN_PRESET = "balanced"  # "quick", "balanced", or "high"

M_DM_MIN_KG = 1.0e-27
M_DM_MAX_KG = 1.0e-20
EPS_MIN = 1.0e-3
EPS_MAX = 1.0e3

# (n_mass, n_epsilon, epsilon chunk size)
RUTHERFORD_SCAN_PRESETS = {
    "quick":    (501, 501, 64),
    "balanced": (1201, 1201, 64),
    "high":     (3001, 3001, 32),
}
N_MASS, N_EPS, EPS_CHUNK_SIZE = RUTHERFORD_SCAN_PRESETS[SCAN_PRESET]

T_DM_K = 300.0
ION_ENERGY_THRESHOLD_J = 1.0e-27
B_MAX_M = 1.0e-3

REPRESENTATIVE_SPEED_QUANTILES = np.array(
    [0.10, 0.30, 0.50, 0.70, 0.90], dtype=float
)
REPRESENTATIVE_SPEED_WEIGHTS = np.full(5, 0.2, dtype=float)

SAVE_FULL_RANGE_RESULTS = True
SCAN_VERSION = "2026-07-19-rutherford-mask-source-v2"
FULL_RANGE_RESULT_NPZ = "rutherford_full_range_energy_scan.npz"
FULL_RANGE_SUMMARY_CSV = "rutherford_full_range_energy_scan_summary.csv"

SHOW_MATHIEU_BOUNDARIES = True
MATHIEU_BOUNDARY_SLOPES_KG = np.array([
    7.533854e-19,
    7.561798e-24,
    7.420566e-26,
    8.939344e-27,
    8.884701e-27,
], dtype=float)

print("Rutherford scan preset:", SCAN_PRESET)
print(f"Grid: {N_MASS} x {N_EPS}; epsilon chunk: {EPS_CHUNK_SIZE}")
print("Mass range [kg]:", M_DM_MIN_KG, "to", M_DM_MAX_KG)
print("Epsilon range:", EPS_MIN, "to", EPS_MAX)


In [ ]:
import constants as c
import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.integrate import solve_ivp

vk = c.vk
vrf = c.vrf
z0 = c.z0
xy1k = c.xy1k
xy2k = c.xy2k
pi = math.pi
um = c.um
cm = c.cm
mm = c.mm
K = c.K
e = c.e
#eps = 1
alpha = 1/137.06
#m_dm = 2.873e-25
m_ion = c.m 
Z_charge = c.Z
freq_x = 719430.7131391969
freq_y = 3031200.0099101723
freq_z = 3002153.5607483205
omega = 2*pi*np.array([freq_x,freq_y,freq_z])
E_THRESHOLD = 1e-27

In [ ]:
def make_area_weighted_b_list(b_max, n_b, focus_power=3.0):
    """Return impact-parameter representatives and exact annular-area weights.

    ``focus_power=1`` reproduces equal-area bins. Values larger than one divide
    the low-b region into more annuli while preserving the exact area measure
    d(b^2)/b_max^2. This is important because rare close encounters dominate
    the high-ion-energy tail.
    """
    if b_max <= 0.0:
        raise ValueError('b_max must be positive')
    if n_b <= 0:
        raise ValueError('n_b must be positive')
    if focus_power <= 0.0:
        raise ValueError('focus_power must be positive')

    s_edges = np.linspace(0.0, 1.0, n_b + 1)
    u_edges = s_edges**focus_power
    u_mid = 0.5 * (u_edges[:-1] + u_edges[1:])

    b_list = b_max * np.sqrt(u_mid)
    b_weights = np.diff(u_edges)
    b_weights /= np.sum(b_weights)
    return b_list, b_weights


In [ ]:
def make_area_weighted_b_list(b_max, n_b, focus_power=3.0):
    """Return impact-parameter representatives with exact annular-area weights."""
    if b_max <= 0.0 or n_b <= 0 or focus_power <= 0.0:
        raise ValueError("b_max, n_b, and focus_power must be positive")
    s_edges = np.linspace(0.0, 1.0, int(n_b) + 1)
    u_edges = s_edges**float(focus_power)
    u_mid = 0.5 * (u_edges[:-1] + u_edges[1:])
    b_values = float(b_max) * np.sqrt(u_mid)
    weights = np.diff(u_edges)
    weights /= weights.sum()
    return b_values, weights


def _rutherford_core(m_dm, eps, speed_dm, b_values):
    """Exact repulsive two-body Rutherford observables.

    This replaces thousands of ODE integrations.  It returns the same physical
    quantities used by the scan: center-of-mass scattering angle, closest
    separation, and final ion recoil energy.
    """
    m_dm = float(m_dm)
    eps = float(eps)
    speed_dm = float(speed_dm)
    b = np.asarray(b_values, dtype=float)

    if m_dm <= 0.0 or speed_dm <= 0.0:
        raise ValueError("m_dm and speed_dm must be positive")
    if eps <= 0.0:
        raise ValueError("This repulsive Rutherford scan requires eps > 0")

    mu = m_dm * m_ion / (m_dm + m_ion)
    E_rel = 0.5 * mu * speed_dm**2
    C = K * abs(Z_charge * eps) * e**2
    a = C / (2.0 * E_rel)

    theta = 2.0 * np.arctan2(a, b)
    r_min = a + np.sqrt(a*a + b*b)

    K_incident = 0.5 * m_dm * speed_dm**2
    transfer_fraction = 4.0 * m_dm * m_ion / (m_dm + m_ion)**2
    E_head_on = transfer_fraction * K_incident
    E_ion = E_head_on / (1.0 + (b / a)**2)

    return theta, r_min, E_ion


def run_rutherford(
    m_dm,
    eps,
    speed_dm,
    plot=False,
    return_full=False,
    b_list=None,
    b_weights=None,
):
    """Fast analytic replacement for the original ODE Rutherford test."""
    if b_list is None or b_weights is None:
        b_list, b_weights = make_area_weighted_b_list(1.0e-3, 200, 3.0)
    b_list = np.asarray(b_list, dtype=float)
    b_weights = np.asarray(b_weights, dtype=float)
    b_weights = b_weights / b_weights.sum()

    thetas, rmins, Edeps = _rutherford_core(
        m_dm, eps, speed_dm, b_list
    )
    detectable = Edeps >= E_THRESHOLD
    rmin_threshold = float(np.max(rmins[detectable])) if np.any(detectable) else np.nan

    if plot:
        fig, ax = plt.subplots(figsize=(7, 5))
        ax.plot(b_list * 1e6, rmins * 1e6, "o-")
        ax.set_xlabel("Impact parameter b [um]")
        ax.set_ylabel("Closest separation [um]")
        ax.set_title("Analytic Rutherford closest approach")
        ax.grid(True)
        fig.tight_layout()
        plt.show()

        fig, ax = plt.subplots(figsize=(7, 5))
        ax.loglog(rmins * 1e6, Edeps, "o")
        ax.axhline(E_THRESHOLD, linestyle="--", label="Threshold")
        ax.set_xlabel("Closest separation [um]")
        ax.set_ylabel("Ion recoil energy [J]")
        ax.set_title("Analytic Rutherford ion recoil")
        ax.legend()
        ax.grid(True, which="both")
        fig.tight_layout()
        plt.show()

        fig, ax = plt.subplots(figsize=(7, 5))
        ax.plot(b_list * 1e6, thetas)
        ax.set_xlabel("Impact parameter b [um]")
        ax.set_ylabel("CM scattering angle [rad]")
        ax.set_title("Analytic Rutherford scattering angle")
        ax.grid(True)
        fig.tight_layout()
        plt.show()

    if return_full:
        return {
            "rmin_threshold": rmin_threshold,
            "b_list": b_list,
            "b_weights": b_weights,
            "rmins": rmins,
            "Edeps": Edeps,
            "thetas": thetas,
            "speed_dm": float(speed_dm),
        }
    return rmin_threshold


In [ ]:
from scipy.stats import maxwell


def most_probable_speed_dm(T, m_dm):
    """Most probable Maxwell--Boltzmann speed."""
    k_B = 1.380649e-23
    return np.sqrt(2.0 * k_B * T / m_dm)


def mb_speed_at_quantile(T, m_dm, quantile):
    """Maxwell--Boltzmann speed percentile."""
    if not (0.0 < quantile < 1.0):
        raise ValueError('quantile must lie between zero and one')
    k_B = 1.380649e-23
    scale = np.sqrt(k_B * T / m_dm)
    return float(maxwell.ppf(quantile, scale=scale))


def representative_MB_speeds(
    T,
    m_dm,
    quantiles=(0.10, 0.30, 0.50, 0.70, 0.90),
):
    """Representative speeds for equal-probability MB bins.

    The default values are the midpoints of five equal-probability bins. Each
    speed therefore carries weight 0.2, and the lowest speed is the 10th
    percentile rather than an extreme 1% or 5% tail.
    """
    quantiles = np.asarray(quantiles, dtype=float)
    if quantiles.ndim != 1 or quantiles.size == 0:
        raise ValueError('quantiles must be a nonempty one-dimensional array')
    if np.any((quantiles <= 0.0) | (quantiles >= 1.0)):
        raise ValueError('all quantiles must lie between zero and one')

    k_B = 1.380649e-23
    scale = np.sqrt(k_B * T / m_dm)
    speeds = maxwell.ppf(quantiles, scale=scale)
    weights = np.full(quantiles.size, 1.0 / quantiles.size)
    return quantiles, speeds, weights


def weighted_quantile(values, quantiles, weights):
    """Weighted empirical quantile with linear interpolation."""
    values = np.asarray(values, dtype=float).reshape(-1)
    weights = np.asarray(weights, dtype=float).reshape(-1)
    quantiles = np.atleast_1d(np.asarray(quantiles, dtype=float))

    good = np.isfinite(values) & np.isfinite(weights) & (weights > 0.0)
    values = values[good]
    weights = weights[good]
    if values.size == 0 or np.sum(weights) <= 0.0:
        out = np.full(quantiles.shape, np.nan)
        return float(out[0]) if out.size == 1 else out

    order = np.argsort(values)
    values = values[order]
    weights = weights[order]
    cumulative = np.cumsum(weights)
    cumulative /= cumulative[-1]
    out = np.interp(quantiles, cumulative, values)
    return float(out[0]) if out.size == 1 else out


def sample_MB_speed(T, m_dm, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    k_B = 1.380649e-23
    sigma_v = np.sqrt(k_B * T / m_dm)
    velocity = rng.normal(0.0, sigma_v, size=3)
    return float(np.linalg.norm(velocity))


# One parameter point

In [ ]:
T = 300.0
m_dm = 2.873e-25
eps = 1.0
speed_quantile = 0.10
speed_dm = mb_speed_at_quantile(T, m_dm, speed_quantile)

out_one_point = run_rutherford(
    m_dm,
    eps,
    speed_dm,
    plot=True,
    return_full=True,
    b_list=make_area_weighted_b_list(1e-3, 120, focus_power=3.0)[0],
    b_weights=make_area_weighted_b_list(1e-3, 120, focus_power=3.0)[1],
)

print(f'MB speed quantile = {speed_quantile:.2f}')
print(f'speed = {speed_dm:.6f} m/s')
print(f'r_min threshold = {out_one_point["rmin_threshold"] * 1e6:.6g} um')
print(
    'Area-weighted mean ion energy = '
    f'{np.average(out_one_point["Edeps"], weights=out_one_point["b_weights"]):.6e} J'
)


# Full-range analytic Rutherford energy scan

The scan uses exact two-body Rutherford formulas.  It does not integrate one ODE
per impact parameter.  The impact-parameter area integral and the conditional
energy quantiles of the five-speed mixture are evaluated analytically.


In [ ]:
from scipy.stats import maxwell
from matplotlib.colors import LogNorm
from matplotlib.ticker import MaxNLocator
import time

K_B = 1.380649e-23


def _log1p_over_x(x):
    x = np.asarray(x, dtype=float)
    out = np.empty_like(x)
    small = np.abs(x) < 1.0e-7
    out[small] = 1.0 - 0.5*x[small] + x[small]**2/3.0
    out[~small] = np.log1p(x[~small]) / x[~small]
    return out


def _survival_at_energy(energy, emax_stack, y_total_stack, weights):
    """Mixture survival P(E_ion >= energy) over representative MB speeds."""
    energy = np.maximum(np.asarray(energy, dtype=float), np.finfo(float).tiny)
    ratio = (emax_stack / energy[None, ...] - 1.0) / y_total_stack
    survival_q = np.clip(ratio, 0.0, 1.0)
    return np.sum(weights[:, None, None] * survival_q, axis=0)


def _mixture_conditional_quantile(
    probability_detectable,
    emax_stack,
    y_total_stack,
    weights,
    threshold,
    quantile,
    iterations=42,
):
    """Exact quantile of the speed-mixture, conditioned on E >= threshold."""
    valid = probability_detectable > 0.0
    lo = np.full(probability_detectable.shape, float(threshold), dtype=float)
    hi = np.max(emax_stack, axis=0)
    hi = np.maximum(hi, lo)
    target = probability_detectable * (1.0 - float(quantile))

    for _ in range(int(iterations)):
        mid = np.sqrt(np.maximum(lo, np.finfo(float).tiny) * np.maximum(hi, np.finfo(float).tiny))
        survival = _survival_at_energy(mid, emax_stack, y_total_stack, weights)
        too_low = survival > target
        lo = np.where(too_low, mid, lo)
        hi = np.where(too_low, hi, mid)

    answer = np.sqrt(lo * hi)
    return np.where(valid, answer, np.nan)


def _add_mathieu_boundaries(ax):
    if not SHOW_MATHIEU_BOUNDARIES:
        return
    eps_line = np.geomspace(EPS_MIN, EPS_MAX, 800)
    for index, slope in enumerate(MATHIEU_BOUNDARY_SLOPES_KG, start=1):
        mass_line = slope * eps_line
        visible = (mass_line >= M_DM_MIN_KG) & (mass_line <= M_DM_MAX_KG)
        if np.any(visible):
            ax.plot(
                eps_line[visible], mass_line[visible], linestyle="--",
                linewidth=1.0, label=f"Mathieu {index}" if index == 1 else None,
            )


def _metric_levels(data, logarithmic=True, max_levels=8):
    finite = np.asarray(data, dtype=float)
    finite = finite[np.isfinite(finite) & ((finite > 0.0) if logarithmic else True)]
    if finite.size == 0 or np.nanmax(finite) <= np.nanmin(finite):
        return np.array([])
    if logarithmic:
        raw = np.geomspace(np.nanmin(finite), np.nanmax(finite), max_levels + 2)[1:-1]
    else:
        raw = MaxNLocator(nbins=max_levels).tick_values(np.nanmin(finite), np.nanmax(finite))
        raw = raw[(raw > np.nanmin(finite)) & (raw < np.nanmax(finite))]
    return np.unique(raw)


def plot_full_range_metric(data, title, label, *, probability=False, scale=1.0):
    z = np.asarray(data, dtype=float) * scale
    finite = z[np.isfinite(z)]
    if finite.size == 0:
        print("No finite values for", title)
        return
    fig, ax = plt.subplots(figsize=(9.0, 6.7))
    if probability:
        mesh = ax.pcolormesh(EPS_GRID, M_GRID, z, shading="auto", vmin=0.0, vmax=1.0, rasterized=True)
        candidate = np.array([1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 0.5])
        levels = candidate[(candidate > np.nanmin(finite)) & (candidate < np.nanmax(finite))]
    else:
        positive = finite[finite > 0.0]
        norm = LogNorm(vmin=np.nanmin(positive), vmax=np.nanmax(positive)) if positive.size else None
        mesh = ax.pcolormesh(EPS_GRID, M_GRID, np.ma.masked_invalid(z), shading="auto", norm=norm, rasterized=True)
        levels = _metric_levels(z, logarithmic=positive.size > 0)
    if levels.size:
        contour = ax.contour(EPS_GRID, M_GRID, z, levels=levels, colors="black", linewidths=0.75)
        ax.clabel(contour, inline=True, fontsize=7, fmt="%.2g")
    _add_mathieu_boundaries(ax)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(r"Charge fraction $\epsilon$")
    ax.set_ylabel(r"DM mass $m_{\rm DM}$ [kg]")
    ax.set_title(title)
    ax.grid(True, which="both", alpha=0.18)
    fig.colorbar(mesh, ax=ax, label=label)
    fig.tight_layout()
    plt.show()


start = time.perf_counter()
m_dm_values = np.geomspace(M_DM_MIN_KG, M_DM_MAX_KG, N_MASS)
eps_values = np.geomspace(EPS_MIN, EPS_MAX, N_EPS)
EPS_GRID, M_GRID = np.meshgrid(eps_values, m_dm_values)

shape = (N_MASS, N_EPS)
output_names = [
    "prob_E_above_threshold",
    "E_unconditional_mean_J",
    "E_yield_above_threshold_J_per_incoming_DM",
    "E_detectable_conditional_mean_J",
    "E_detectable_conditional_median_J",
    "E_detectable_conditional_p90_J",
    "E_detectable_conditional_p99_J",
    "r_min_threshold_m",
    "q10_prob_E_above_threshold",
    "q10_E_b_weighted_mean_J",
    "q10_E_detectable_conditional_median_J",
    "q10_E_detectable_conditional_p90_J",
    "q10_r_min_threshold_m",
]
outputs = {name: np.full(shape, np.nan, dtype=np.float32) for name in output_names}

speed_units = maxwell.ppf(REPRESENTATIVE_SPEED_QUANTILES)
weights = REPRESENTATIVE_SPEED_WEIGHTS.astype(float)
B2 = B_MAX_M**2

for j0 in range(0, N_EPS, EPS_CHUNK_SIZE):
    j1 = min(j0 + EPS_CHUNK_SIZE, N_EPS)
    m = m_dm_values[:, None]
    eps = eps_values[None, j0:j1]
    block_shape = (N_MASS, j1 - j0)

    transfer = 4.0 * m * m_ion / (m + m_ion)**2
    emax_list = []
    y_total_list = []
    probability_list = []
    mean_list = []
    yield_list = []
    conditional_mean_list = []
    conditional_median_list = []
    conditional_p90_list = []
    r_threshold_list = []

    for speed_unit in speed_units:
        v2 = speed_unit**2 * K_B * T_DM_K / m
        mu = m * m_ion / (m + m_ion)
        E_rel = 0.5 * mu * v2
        C = K * abs(Z_charge) * e**2 * eps
        a = C / (2.0 * E_rel)
        y_total = B2 / np.maximum(a*a, np.finfo(float).tiny)

        K_incident = 0.5 * speed_unit**2 * K_B * T_DM_K
        E_max = transfer * K_incident
        y_detect = np.clip(E_max / ION_ENERGY_THRESHOLD_J - 1.0, 0.0, y_total)
        prob_q = np.divide(y_detect, y_total, out=np.zeros(block_shape), where=y_total > 0.0)

        mean_q = E_max * _log1p_over_x(y_total)
        yield_q = E_max * np.divide(
            np.log1p(y_detect), y_total,
            out=np.zeros(block_shape), where=y_total > 0.0,
        )
        cond_mean_q = np.where(y_detect > 0.0, E_max * _log1p_over_x(y_detect), np.nan)
        cond_median_q = np.where(y_detect > 0.0, E_max / (1.0 + 0.5*y_detect), np.nan)
        cond_p90_q = np.where(y_detect > 0.0, E_max / (1.0 + 0.1*y_detect), np.nan)
        r_thr_q = np.where(y_detect > 0.0, a * (1.0 + np.sqrt(1.0 + y_detect)), np.nan)

        emax_list.append(np.broadcast_to(E_max, block_shape))
        y_total_list.append(y_total)
        probability_list.append(prob_q)
        mean_list.append(mean_q)
        yield_list.append(yield_q)
        conditional_mean_list.append(cond_mean_q)
        conditional_median_list.append(cond_median_q)
        conditional_p90_list.append(cond_p90_q)
        r_threshold_list.append(r_thr_q)

    emax_stack = np.stack(emax_list)
    y_total_stack = np.stack(y_total_list)
    probability_stack = np.stack(probability_list)
    mean_stack = np.stack(mean_list)
    yield_stack = np.stack(yield_list)
    r_threshold_stack = np.stack(r_threshold_list)

    probability = np.sum(weights[:, None, None] * probability_stack, axis=0)
    unconditional_mean = np.sum(weights[:, None, None] * mean_stack, axis=0)
    energy_yield = np.sum(weights[:, None, None] * yield_stack, axis=0)
    conditional_mean = np.divide(
        energy_yield, probability,
        out=np.full(block_shape, np.nan), where=probability > 0.0,
    )
    conditional_median = _mixture_conditional_quantile(
        probability, emax_stack, y_total_stack, weights,
        ION_ENERGY_THRESHOLD_J, 0.50,
    )
    conditional_p90 = _mixture_conditional_quantile(
        probability, emax_stack, y_total_stack, weights,
        ION_ENERGY_THRESHOLD_J, 0.90,
    )
    conditional_p99 = _mixture_conditional_quantile(
        probability, emax_stack, y_total_stack, weights,
        ION_ENERGY_THRESHOLD_J, 0.99,
    )
    r_threshold = np.nanmax(r_threshold_stack, axis=0)
    r_threshold[~np.any(np.isfinite(r_threshold_stack), axis=0)] = np.nan

    q10_prob = probability_stack[0]
    q10_mean = mean_stack[0]
    q10_cond_median = np.stack(conditional_median_list)[0]
    q10_cond_p90 = np.stack(conditional_p90_list)[0]
    q10_r = r_threshold_stack[0]

    block_values = {
        "prob_E_above_threshold": probability,
        "E_unconditional_mean_J": unconditional_mean,
        "E_yield_above_threshold_J_per_incoming_DM": energy_yield,
        "E_detectable_conditional_mean_J": conditional_mean,
        "E_detectable_conditional_median_J": conditional_median,
        "E_detectable_conditional_p90_J": conditional_p90,
        "E_detectable_conditional_p99_J": conditional_p99,
        "r_min_threshold_m": r_threshold,
        "q10_prob_E_above_threshold": q10_prob,
        "q10_E_b_weighted_mean_J": q10_mean,
        "q10_E_detectable_conditional_median_J": q10_cond_median,
        "q10_E_detectable_conditional_p90_J": q10_cond_p90,
        "q10_r_min_threshold_m": q10_r,
    }
    for name, value in block_values.items():
        outputs[name][:, j0:j1] = np.asarray(value, dtype=np.float32)

    print(f"epsilon columns {j0:5d}:{j1:5d} / {N_EPS}")

runtime_s = time.perf_counter() - start
print(f"Full analytic Rutherford scan finished in {runtime_s:.2f} s")
print("Lowest representative speed quantile: q=0.10")

if SAVE_FULL_RANGE_RESULTS:
    np.savez_compressed(
        FULL_RANGE_RESULT_NPZ,
        scan_version=np.asarray(SCAN_VERSION),
        m_dm_values=m_dm_values,
        eps_values=eps_values,
        temperature_K=np.asarray(T_DM_K),
        ion_energy_threshold_J=np.asarray(ION_ENERGY_THRESHOLD_J),
        b_max_m=np.asarray(B_MAX_M),
        ion_mass_kg=np.asarray(m_ion),
        ion_charge_number=np.asarray(Z_charge),
        speed_quantiles=REPRESENTATIVE_SPEED_QUANTILES,
        speed_weights=REPRESENTATIVE_SPEED_WEIGHTS,
        runtime_s=runtime_s,
        **outputs,
    )
    summary = pd.DataFrame({
        "metric": list(outputs),
        "finite_count": [int(np.count_nonzero(np.isfinite(outputs[k]))) for k in outputs],
        "minimum": [float(np.nanmin(outputs[k])) if np.any(np.isfinite(outputs[k])) else np.nan for k in outputs],
        "maximum": [float(np.nanmax(outputs[k])) if np.any(np.isfinite(outputs[k])) else np.nan for k in outputs],
    })
    summary.to_csv(FULL_RANGE_SUMMARY_CSV, index=False)
    print("Saved", FULL_RANGE_RESULT_NPZ)
    print("This file is the mask source for the potential notebook.")
    print("Saved", FULL_RANGE_SUMMARY_CSV)

plot_full_range_metric(
    outputs["prob_E_above_threshold"],
    "Rutherford probability of ion recoil above threshold",
    "Probability", probability=True,
)
plot_full_range_metric(
    outputs["E_detectable_conditional_median_J"],
    "Conditional median ion recoil energy",
    "Energy [J]",
)
plot_full_range_metric(
    outputs["E_detectable_conditional_p90_J"],
    "Conditional p90 ion recoil energy",
    "Energy [J]",
)
plot_full_range_metric(
    outputs["E_unconditional_mean_J"],
    "Unconditional mean ion recoil per incoming DM",
    "Energy [J per incoming DM]",
)
plot_full_range_metric(
    outputs["q10_E_b_weighted_mean_J"],
    "10th-percentile MB speed: area-weighted mean ion recoil",
    "Energy [J]",
)
plot_full_range_metric(
    outputs["r_min_threshold_m"],
    "Largest closest approach producing threshold recoil",
    r"$r_{\min}$ [$\mu$m]", scale=1.0e6,
)


In [ ]:
def evaluate_custom_point(
    m_dm_custom,
    eps_custom,
    speed_custom=None,
    speed_quantile=0.10,
    T_dm=300.0,
    plot=False,
    b_max=1e-3,
    n_b=120,
    b_focus_power=3.0,
):
    """Evaluate one point using q10 by default.

    Pass ``speed_custom`` to override the Maxwell--Boltzmann percentile.
    """
    if speed_custom is None:
        speed_custom = mb_speed_at_quantile(
            T_dm,
            m_dm_custom,
            speed_quantile,
        )
        speed_source = f'MB q={speed_quantile:.2f}'
    else:
        speed_custom = float(speed_custom)
        speed_source = 'user supplied'

    b_list_custom, b_weights_custom = make_area_weighted_b_list(
        b_max=b_max,
        n_b=n_b,
        focus_power=b_focus_power,
    )
    out = run_rutherford(
        m_dm=m_dm_custom,
        eps=eps_custom,
        speed_dm=speed_custom,
        plot=plot,
        return_full=True,
        b_list=b_list_custom,
        b_weights=b_weights_custom,
    )

    Edeps = np.asarray(out['Edeps'], dtype=float)
    rmins = np.asarray(out['rmins'], dtype=float)
    weights = np.asarray(out['b_weights'], dtype=float)
    weights /= np.sum(weights)
    detectable = Edeps >= E_THRESHOLD

    probability = float(np.sum(weights[detectable]))
    mean_energy = float(np.average(Edeps, weights=weights))
    median_energy = weighted_quantile(Edeps, 0.50, weights)

    if np.any(detectable):
        conditional_median = weighted_quantile(
            Edeps[detectable], 0.50, weights[detectable]
        )
        conditional_p90 = weighted_quantile(
            Edeps[detectable], 0.90, weights[detectable]
        )
        rmin_threshold = float(np.max(rmins[detectable]))
    else:
        conditional_median = np.nan
        conditional_p90 = np.nan
        rmin_threshold = np.nan

    print('Custom Rutherford point:')
    print(f'm_dm       = {m_dm_custom:.6e} kg')
    print(f'eps        = {eps_custom:.6e}')
    print(f'speed      = {speed_custom:.6f} m/s ({speed_source})')
    print(f'P detectable over b = {probability:.6e}')
    print(f'unconditional mean ion energy   = {mean_energy:.6e} J')
    print(f'unconditional median ion energy = {median_energy:.6e} J')
    print(f'conditional median detectable E = {conditional_median:.6e} J')
    print(f'conditional p90 detectable E    = {conditional_p90:.6e} J')
    print(f'r_min threshold                = {rmin_threshold * 1e6:.6g} um')

    return {
        'm_dm_kg': m_dm_custom,
        'eps': eps_custom,
        'speed_source': speed_source,
        'speed_quantile': speed_quantile if speed_source.startswith('MB') else np.nan,
        'speed_dm_m_s': speed_custom,
        'prob_E_above_threshold': probability,
        'E_unconditional_mean_J': mean_energy,
        'E_unconditional_median_J': median_energy,
        'E_detectable_conditional_median_J': conditional_median,
        'E_detectable_conditional_p90_J': conditional_p90,
        'r_min_threshold_m': rmin_threshold,
        'r_min_threshold_um': rmin_threshold * 1e6 if np.isfinite(rmin_threshold) else np.nan,
        'full_output': out,
    }


m_dm_custom = 2.637e-25
eps_custom = 2.336

# No manual speed is needed: this uses the 10th-percentile MB speed.
custom_result = evaluate_custom_point(
    m_dm_custom=m_dm_custom,
    eps_custom=eps_custom,
    speed_quantile=0.10,
    speed_custom=None,
    T_dm=300.0,
    plot=True,
)
